In [6]:
%load_ext dotenv
%dotenv

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [7]:
import os

In [8]:
# for key, val in os.environ.items():
#     print(f"{key}: {val}")

### Doc2Text

In [9]:
from langchain_community.document_loaders import Docx2txtLoader
import copy

In [10]:
loader_docx = Docx2txtLoader('Introduction_to_Data_and_Data_Science.docx')

In [11]:
pages_docx = loader_docx.load()

In [12]:
pages_docx

[Document(metadata={'source': 'Introduction_to_Data_and_Data_Science.docx'}, page_content="Analysis vs Analytics\n\nAlright! So…\nLet’s discuss the not-so-obvious differences\nbetween the terms analysis and analytics.\nDue to the similarity of the words, some people\nbelieve they share the same meaning, and thus\nuse them interchangeably. Technically, this\nisn’t correct. There is, in fact, a distinct\ndifference between the two. And the reason\nfor one often being used instead of the other\nis the lack of a transparent understanding\nof both.\nSo, let’s clear this up, shall we?\nFirst, we will start with analysis.\nConsider the following…\nYou have a huge dataset containing data of\nvarious types. Instead of tackling the entire\ndataset and running the risk of becoming overwhelmed,\nyou separate it into easier to digest chunks\nand study them individually and examine how\nthey relate to other parts. And that’s analysis\nin a nutshell.\nOne important thing to remember, however,\nis tha

In [13]:
pages_docx_cut = copy.deepcopy(pages_docx)

In [14]:
for i in pages_docx_cut:
    i.page_content = ' '.join(pages_docx_cut[0].page_content.split())

In [15]:
from langchain_text_splitters.character import CharacterTextSplitter

In [16]:
char_splitter = CharacterTextSplitter(separator= "", 
                                      chunk_size=500, 
                                      chunk_overlap = 50)

In [17]:
pages_char_list = char_splitter.split_documents(pages_docx_cut)

In [18]:
print(pages_char_list[0].page_content, "\n\n", pages_char_list[1].page_content)

Analysis vs Analytics Alright! So… Let’s discuss the not-so-obvious differences between the terms analysis and analytics. Due to the similarity of the words, some people believe they share the same meaning, and thus use them interchangeably. Technically, this isn’t correct. There is, in fact, a distinct difference between the two. And the reason for one often being used instead of the other is the lack of a transparent understanding of both. So, let’s clear this up, shall we? First, we will star 

 let’s clear this up, shall we? First, we will start with analysis. Consider the following… You have a huge dataset containing data of various types. Instead of tackling the entire dataset and running the risk of becoming overwhelmed, you separate it into easier to digest chunks and study them individually and examine how they relate to other parts. And that’s analysis in a nutshell. One important thing to remember, however, is that you perform analyses on things that have already happened in

### Embedding

In [19]:
!pip install sentence-transformers

In [20]:
from sentence_transformers import SentenceTransformer
import torch

# Load a pretrained sentence transformer model
model_name = "all-MiniLM-L6-v2"
model = SentenceTransformer(model_name)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [18]:
v2 = model.encode(
    pages_char_list[0].page_content,
    convert_to_tensor=True,   # returns torch tensor
    normalize_embeddings=True # useful for cosine similarity
)

In [29]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

C:\Users\AbbasKothari\AppData\Local\Temp\ipykernel_11164\3744977501.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Vector Store

In [22]:
!pip install chromadb

   ---------------------------------------- 0.0/21.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/21.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/21.9 MB ? eta -:--:--
   ---------------------------------------- 0.3/21.9 MB ? eta -:--:--
   ---------------------------------------- 0.3/21.9 MB ? eta -:--:--
   ---------------------------------------- 0.3/21.9 MB ? eta -:--:--
    --------------------------------------- 0.5/21.9 MB 542.8 kB/s eta 0:00:40
    --------------------------------------- 0.5/21.9 MB 542.8 kB/s eta 0:00:40
   - -------------------------------------- 0.8/21.9 MB 493.1 kB/s eta 0:00:43
   - -------------------------------------- 0.8/21.9 MB 493.1 kB/s eta 0:00:43
   - -------------------------------------- 0.8/21.9 MB 493.1 kB/s eta 0:00:43
   - -------------------------------------- 1.0/21.9 MB 463.3 kB/s eta 0:00:45
   - -------------------------------------- 1.0/21.9 MB 463.3 kB/s eta 0:00:45
   - ----------------------

In [23]:
from langchain_community.vectorstores import Chroma

In [30]:
vectorstore = Chroma.from_documents(
    documents= pages_char_list,
    embedding= embedding,
    persist_directory= "./vectordb"
)